# ML05 · KMeans 聚类

用最小范例创建对非监督聚类的直觉：让 KMeans 自己把没有标签的数据分成几群，并学会用肘部法挑选群数。

## 学习目标
- 理解聚类（clustering）与非监督学习（unsupervised learning）的定位：没有标准答案，靠数据结构自己分组。
- 创建 KMeans 的直觉：质心（centroid）与「分配点、移动质心」反复迭代的过程。
- 会用 sklearn 的 KMeans，掌握 fit_predict、labels_、cluster_centers_ 三件事。
- 认识惯性（inertia）的意义，并用肘部法（elbow method）决定群数 k。
- 能把聚类结果依群别上色，做出可解读的可视化。

In [ ]:
# 概念：画图前的共用设置，让中文标题正常显示、固定乱数种子方便对照
import numpy as np
import matplotlib.pyplot as plt

# 让 matplotlib 能显示繁体中文，避免标题出现方框
plt.rcParams["font.sans-serif"] = ["Microsoft JhengHei", "PingFang TC", "DFKai-SB", "sans-serif"]
plt.rcParams["axes.unicode_minus"] = False   # 负号用 ASCII，避免显示异常

print("环境设置完成")

## 什么是聚类与非监督学习

聚类（clustering）是把一堆数据依「彼此有多像」自动分成几组（群，cluster）的工作。它属于非监督学习（unsupervised learning）：训练数据只有特征、没有标签（label），模型要自己从数据结构找出分组。

为什么要学：前面的分类（classification）是监督学习（supervised learning），需要每笔数据都附正确答案；但真实情境常常没有答案，例如想把一批街区依型态分组却没人事先标好。这时就靠聚类让数据自己说话。

差别一句话：监督学习是「给答案、学对应」；非监督学习是「没答案、找结构」。

In [ ]:
# 概念：非监督数据只有特征、没有标签；肉眼能看出几团，正是聚类想自动做到的事
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)   # 固定乱数种子，让每次运行的散点一致

# 造三团二维假数据：每团在不同中心附近散开（例如三种型态的街区）
center_a = rng.normal(loc=[2, 2], scale=0.5, size=(30, 2))
center_b = rng.normal(loc=[8, 3], scale=0.5, size=(30, 2))
center_c = rng.normal(loc=[5, 8], scale=0.5, size=(30, 2))

points = np.vstack([center_a, center_b, center_c])   # 三团叠成一份数据，共 90 个点

# 注意：此处刻意不保留各点来自哪一团的信息，仿真「没有标签」的真实情境
print("数据 shape:", points.shape)   # (90, 2)：90 个样本、每个 2 维

plt.figure(figsize=(5, 4))
plt.scatter(points[:, 0], points[:, 1], color="gray")   # 全部同色，因为我们还不知道分组
plt.title("没有标签的二维点：肉眼看似三团")
plt.xlabel("特征 1")
plt.ylabel("特征 2")
plt.show()

## KMeans 直觉：质心与迭代

KMeans 的核心是质心（centroid）：每一群用一个中心点代表。算法不靠复杂公式，而是反复做两件事直到稳定：
1. 分配：把每个点分到「离它最近的质心」那一群。
2. 更新：把每群的质心移到该群所有点的平均位置。

为什么会收敛（convergence）：每次迭代（iteration）质心都往群的中央靠，群内距离越来越小，移动幅度越来越小，最后几乎不动就停。理解这个循环，就能明白 KMeans 在做的只是「反复逼近群中心」。

In [ ]:
# 概念：手动跑几轮「分配点、更新质心」，观察质心如何从随机位置移到群中央
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(1)

# 造两团容易分开的点当示范
group1 = rng.normal(loc=[2, 2], scale=0.6, size=(40, 2))
group2 = rng.normal(loc=[7, 7], scale=0.6, size=(40, 2))
points = np.vstack([group1, group2])

# 随机挑两个起始质心（故意放在不太准的位置，看它怎么移动）
centroids = np.array([[1.0, 6.0], [6.0, 2.0]])

for step in range(3):                                   # 只跑前三轮，足以看出趋势
    # 分配：算每个点到两个质心的距离，取较近的那群（0 或 1）
    dist = np.linalg.norm(points[:, None, :] - centroids[None, :, :], axis=2)
    assign = dist.argmin(axis=1)                        # 每个点所属的群编号

    # 更新：把每个质心移到该群所有点的平均位置
    new_centroids = np.array([points[assign == k].mean(axis=0) for k in range(2)])

    print(f"第 {step+1} 轮后质心:\n", np.round(new_centroids, 2))
    centroids = new_centroids   # 用更新后的质心进入下一轮

plt.figure(figsize=(5, 4))
plt.scatter(points[:, 0], points[:, 1], c=assign, cmap="viridis")
plt.scatter(centroids[:, 0], centroids[:, 1], color="red", marker="X", s=200)   # 红叉为最后质心
plt.title("手动迭代：质心移到群中央")
plt.show()

## 动手用 sklearn KMeans

实务上不必自己写迭代，sklearn 的 KMeans 对象帮我们处理好。使用时设置群数 n_clusters，再让它从数据学出聚类。

三个常用输出：
- fit_predict：一次完成「学习 + 回传每个点的群编号」。
- labels_：每个样本被分到的群编号（与输入数据一一对应）。
- cluster_centers_：各群质心的座标。

形状：`KMeans(n_clusters=群数).fit_predict(数据)`。

In [ ]:
# 概念：用 sklearn KMeans 对多团数据聚类，取得 labels_ 与 cluster_centers_
import numpy as np
from sklearn.cluster import KMeans

rng = np.random.default_rng(0)

# 造三团二维数据（与前面同样的造法，确保自足）
center_a = rng.normal(loc=[2, 2], scale=0.5, size=(30, 2))
center_b = rng.normal(loc=[8, 3], scale=0.5, size=(30, 2))
center_c = rng.normal(loc=[5, 8], scale=0.5, size=(30, 2))
points = np.vstack([center_a, center_b, center_c])

# n_init 设明确值避免版本差异的警告；random_state 固定让结果可重现
kmeans = KMeans(n_clusters=3, n_init=10, random_state=0)
labels = kmeans.fit_predict(points)   # 一步到位：学习并回传每个点的群编号

print("前 10 个点的群编号:", labels[:10])
print("labels_ 是否等于 fit_predict 回传值:", np.array_equal(labels, kmeans.labels_))
print("三个群的质心座标:\n", np.round(kmeans.cluster_centers_, 2))
# 与肉眼预期对照：三个质心应分别落在 (2,2)、(8,3)、(5,8) 附近
print("各群点数:", np.bincount(labels))

## 选几群才对：inertia 与肘部法

群数 k 要自己决定，而 KMeans 用惯性（inertia）衡量聚类的紧密程度：它是「每个点到所属质心距离平方的总和」，越小代表群内越聚集。

问题是：k 越大 inertia 一定越小（极端情况每个点自成一群、inertia 为 0），所以不能只看谁小。肘部法（elbow method）的做法是画出 inertia 对 k 的曲线，找曲线从「陡降」转为「平缓」的转折点（手肘），那个 k 通常就是合理群数。

惯性可由 `kmeans.inertia_` 取得。

In [ ]:
# 概念：对 k 从 1 到 8 各跑一次 KMeans，搜集 inertia 画肘部法曲线
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans

rng = np.random.default_rng(0)
center_a = rng.normal(loc=[2, 2], scale=0.5, size=(30, 2))
center_b = rng.normal(loc=[8, 3], scale=0.5, size=(30, 2))
center_c = rng.normal(loc=[5, 8], scale=0.5, size=(30, 2))
points = np.vstack([center_a, center_b, center_c])

k_values = range(1, 9)
inertias = []
for k in k_values:
    km = KMeans(n_clusters=k, n_init=10, random_state=0)
    km.fit(points)                  # 这里只需要学习以取得 inertia_，不必拿回 labels
    inertias.append(km.inertia_)    # 累积每个 k 的群内离散程度

print("各 k 的 inertia:", np.round(inertias, 1))

plt.figure(figsize=(5, 4))
plt.plot(list(k_values), inertias, marker="o")
plt.title("肘部法：inertia 随 k 变化")
plt.xlabel("群数 k")
plt.ylabel("inertia（群内离散）")
# 技巧：手肘大约在曲线「明显转平」的 k；本例数据有三团，预期手肘落在 k=3
plt.show()

## 聚类结果可视化与解读

聚类没有标准答案，所以可视化是检查结果合不合理的主要手段（视觉检查，visual sanity check）。做法是把点依群别上色（color by label），再把质心叠上去。

看两件事：
- 群是否分得开：不同颜色的点有没有明显界线，还是混在一起。
- 质心是否落在群中央：红色标记应该位于各色点团的中心。

若颜色乱成一团或质心偏离中心，通常代表 k 选得不对或特征尺度有问题。

In [ ]:
# 概念：依 labels_ 上色、叠上 cluster_centers_，一眼判读聚类品质
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans

rng = np.random.default_rng(0)
center_a = rng.normal(loc=[2, 2], scale=0.5, size=(30, 2))
center_b = rng.normal(loc=[8, 3], scale=0.5, size=(30, 2))
center_c = rng.normal(loc=[5, 8], scale=0.5, size=(30, 2))
points = np.vstack([center_a, center_b, center_c])

kmeans = KMeans(n_clusters=3, n_init=10, random_state=0)
labels = kmeans.fit_predict(points)
centers = kmeans.cluster_centers_

plt.figure(figsize=(5, 4))
# c=labels 让同群同色；cmap 指定配色，不同群自动拿到不同颜色
plt.scatter(points[:, 0], points[:, 1], c=labels, cmap="viridis")
plt.scatter(centers[:, 0], centers[:, 1], color="red", marker="X", s=200)   # 质心用红叉标出
plt.title("聚类结果：依群别上色并标记质心")
plt.xlabel("特征 1")
plt.ylabel("特征 2")
plt.show()

# 解读依据：三群颜色分得开、红叉落在各团中央，代表聚类合理
print("各群点数:", np.bincount(labels))

## 练习

以下三题由浅入深，皆为复合型但简短。数据请自己用 numpy 造（仿真即可），完成后对照「验收」确认输出。

In [ ]:
# TODO 1 ·（简单）楼高分两群（集成：聚类 + fit_predict + labels_）
#   情境：你有一批建筑的楼高（floor height）数值，混有低矮房与高楼两种，但没有标好哪栋属于哪类。
#   要求：
#     1. 用 numpy 造两团不同高度的楼高数据（例如一团在 10 公尺附近、一团在 50 公尺附近），合并成一份一维数据。
#     2. 把数据 reshape 成 (N, 1)，用 KMeans 设 n_clusters=2 做 fit_predict 取得 labels_，并印出各群点数与两个质心。
#   验收：应该看到数据被分成「低楼群」与「高楼群」两群，两个质心分别落在低、高楼高附近，分界大致符合直觉。
# TODO: 在这里完成


In [ ]:
# TODO 2 ·（中间）街区聚类与群数选择（集成：质心迭代直觉 + inertia + 肘部法 + cluster_centers_）
#   情境：你为每个街区（block）记录「楼高、基地面积」两维数据，城市里有三种型态的街区，但没有人事先分类。
#   要求：
#     1. 用 numpy 造三团二维街区数据（各自围绕不同中心）并合并。
#     2. 对 k 从 1 到 6 各跑一次 KMeans，记录每个 inertia（用 kmeans.inertia_）。
#     3. 画肘部法曲线挑出合理 k，再用该 k 重新 fit 一次取得 cluster_centers_。
#   验收：应该看到 inertia 曲线在 k=3 附近出现手肘，且三个质心分别落在三团数据中央。
# TODO: 在这里完成


In [ ]:
# TODO 3 ·（稍难）容积率聚类的尺度陷阱（集成：聚类 + inertia + 可视化 + 特征尺度思考）
#   情境：你为街区记录「楼高（数十公尺）」与「容积率 floor area ratio（小数，约 0 到 5）」两维数据，
#         两个特征的数值范围差很大。
#   要求：
#     1. 用 numpy 造这两维数据（楼高量级为数十、容积率为小数），直接对原始数据做 KMeans 并依群别上色可视化。
#     2. 将两个特征各自做 z-score 标准化（减平均、除标准差）后，再做一次 KMeans 并可视化。
#     3. 比较两次聚类的 labels 与 inertia，并用文字说明为何尺度差异会让某一维（楼高）主导聚类。
#   验收：应该看到未标准化时聚类几乎只跟着楼高走（界线沿楼高方向切），
#         标准化后两个特征才一起影响聚类结果。
# TODO: 在这里完成


## 小结

- 聚类属于非监督学习：数据只有特征、没有标签，模型靠数据结构自己分组，与需要答案的分类形成对照。
- KMeans 的核心是反复做「把点分到最近质心、再把质心移到群平均」，迭代到质心几乎不动就收敛，不必硬背公式。
- sklearn 的 KMeans 用 n_clusters 设群数，fit_predict 一次取得聚类，labels_ 是每点群编号、cluster_centers_ 是各群质心座标。
- inertia 衡量群内离散程度，会随 k 变大而单调下降；肘部法看 inertia 对 k 曲线的转折点挑合理 k。
- 聚类没有正确答案，依群别上色并叠上质心做视觉检查，看群是否分得开、质心是否落在中央。
- 特征尺度差很大时，数值大的特征会主导距离计算与聚类结果，先标准化能让各特征公平地一起影响聚类。